<a href="https://colab.research.google.com/github/patriciasandagorda/UA_MDM_Labo2_Grupo2/blob/ramaPatricia/06_kaggle_texto_optuna_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este script permite descargar datos de una competencia de Kaggle directamente en Google Colab.

Requiere que el usuario genere un token de autenticación de Kaggle (kaggle.json)
y lo suba al entorno de Colab para autenticar la descarga.

Guía para generar el token:
1. Ir a https://www.kaggle.com
2. En la sección 'API', hacer clic en 'Create New API Token'
3. Esto descargará un archivo llamado `kaggle.json` en tu computadora
4. Luego, en Colab, deberás subir este archivo como se muestra en las instrucciones siguientes

Guía para Generar token y descargar: https://www.kaggle.com/discussions/general/74235.


In [ ]:
# Instala librerías
!pip install optuna
!pip install -q kaggle

from google.colab import files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 16.4 MB/s eta 0:00:00


In [ ]:
# Cargar el token de Kaggle (json)
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"patriciasandagorda","key":"3a302257a5acb1c7824a66f353cdcef6"}'}

In [ ]:
 # Crea el directorio de configuración de Kaggle si no existe
 ! mkdir ~/.kaggle

 # Copia el archivo `kaggle.json` al directorio de configuración
 ! cp kaggle.json ~/.kaggle/

 # Establece los permisos de acceso al file
 ! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Lista datasets públicos disponibles (a modo de validación)
! kaggle datasets list

ref                                                                 title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
algozee/teenager-menthal-healy                                      Social Media Impact on Teen Mental Health                16190  2026-04-05 08:04:21.823000          15465        337                1  
meruvakodandasuraj/e-commerce-customer-behavior-and-sales-20202026  E-Commerce Customer Behavior & Sales 2020–2026         1370061  2026-04-22 15:58:06.760000           1521         30                1  
syedaeman2212/shoes-sales-dataset                                   Shoes Sales Dataset                                      18813  2026-04-23 16:28:33.087000           1078         29

In [ ]:
# Descarga los datos de la competencia
! kaggle competitions download -c 'petfinder-adoption-prediction'

100% 1.94G/1.94G [00:55<00:00, 37.7MB/s]



In [ ]:
# Crea la carpeta input
! mkdir input
# Crea un subcarpeta de la competencia
! mkdir ./input/petfinder-adoption-prediction
# Mueve el archivo ZIP descargado a la carpeta input
! mv petfinder-adoption-prediction.zip ./input
# Descomprime el archivo ZIP
! unzip ./input/petfinder-adoption-prediction.zip -d ./input/petfinder-adoption-prediction/

Se truncaron las últimas líneas 5000 del resultado de transmisión.
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a89bfa0aa.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a89d4e8f3.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a89f8b241.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a89fd1f1e.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a8a1d4151.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a8a2aecc3.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a8a3c4f49.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a8a8f95f6.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a8b358af2.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a8b9d57b4.json  
  inflating: ./input/petfinder-adoption-prediction/train_sentiment/a8ba4dfa1.json  
  inflati

In [ ]:
#06_text_classification

Fuentes: https://medium.com/nlplanet/fine-tuning-distilbert-on-senator-tweets-a6f2425ca50e

#### **Instalar Modulos**

conda install datasets=="2.20.0"

conda install transformers=="4.40.1"

conda install numpy=="1.26.4" # La última versión no funciona bien


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import copy

import time
import datetime

from sklearn.metrics import confusion_matrix, cohen_kappa_score

from datasets import Dataset, DatasetDict
#from UA_MDM_LDI_II.tutoriales.utils import plot_confusion_matrix

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

# Modeling
import torch
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast, DataCollatorWithPadding, AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW # Corrected import for AdamW

# Progress bar
from tqdm.auto import tqdm

# from utils import plot_confusion_matrix, get_artifact_filename # Comentado debido a ModuleNotFoundError

# Placeholder for get_artifact_filename - REMOVE THIS ONCE utils.py IS AVAILABLE
def get_artifact_filename(study, artifact_type):
    """
    Placeholder function for the missing 'get_artifact_filename' from utils.py.
    This attempts to construct a filename based on common Optuna artifact saving patterns.
    It assumes the artifact is associated with a specific study.
    If the study has a best trial (e.g., from a loaded existing study), it tries to use its number.
    Otherwise, it defaults to a generic filename. This might lead to FileNotFoundError
    if the actual artifact name differs or the file doesn't exist.
    """
    study_name_for_filename = study.study_name.replace(' ', '_').replace('-', '_')
    try:
        if study.best_trial:
            # Assuming artifacts are saved with trial numbers like 'artifact_type_study_name_trial_number.joblib'
            return f"{artifact_type}_{study_name_for_filename}_{study.best_trial.number}.joblib"
        else:
            # This else branch might not be reached if best_trial raises ValueError instead of returning None
            return f"{artifact_type}_{study_name_for_filename}_fixed.joblib"
    except ValueError:
        # If ValueError occurs (no best trial), fallback to a fixed filename
        return f"{artifact_type}_{study_name_for_filename}_fixed.joblib"


from joblib import load, dump

# Verificamos que CUDA está funcional
torch.cuda.is_available()

True

**Bajamos el modelo**

In [ ]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')  #19.11   5 categorias disntintas

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Armado de los Datasets**

In [ ]:
# Paths
BASE_DIR = '/content/'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

# Parametros y variables
SEED = 42
TEST_SIZE = 0.2
BATCH_SIZE = 64

MODEL_NAME = '06 Bert'

MODEL_VERSION = '1.0'

In [ ]:
# Cargar los datos
df = pd.read_csv(PATH_TO_TRAIN)
df = df[df['Description'].notnull()] #que pasa con los datos que tienen la descripcion nula?
df['labels'] = df["AdoptionSpeed"]


In [ ]:
import os

# Crear el directorio 'work' si no existe
os.makedirs(os.path.dirname(PATH_TO_OPTUNA_ARTIFACTS), exist_ok=True)

In [ ]:


# Dividir los datos usando sklearn
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df.AdoptionSpeed)

# study_lgb = optuna.create_study(direction='maximize',
#                              storage=f"sqlite:///{os.path.join(os.path.dirname(PATH_TO_OPTUNA_ARTIFACTS), 'db.sqlite3')}",  # Specify the storage URL here.
#                              study_name="04-LGB Multiclass CV",
#                             load_if_exists = True)


# lgb_test_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))

# train_df = df[~df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)
# test_df = df[df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)

# Convertir a Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Combinar en un DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'val': test_dataset
})

# Codificar la columna de etiquetas como clases
dataset = dataset.class_encode_column('labels')

# Hacer una lista de columnas para remover antes de la tokenización
cols_to_remove = [col for col in dataset["train"].column_names if col != 'labels']
print(cols_to_remove)

Stringifying the column:   0%|          | 0/11984 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/11984 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/2996 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/2996 [00:00<?, ? examples/s]

['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed', '__index_level_0__']


In [ ]:
# Obtener el objeto ClassLabel del conjunto de datos de entrenamiento
class_label = dataset["train"].features["labels"] # donde se usa???

# Obtener las clases originales a partir del objeto ClassLabel
classes = class_label.names
classes

['0', '1', '2', '3', '4']

In [ ]:
# Tokenize and encode the dataset
def tokenize(batch):
    from transformers import DistilBertTokenizerFast
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    tokenized_batch = tokenizer(batch["Description"], padding=True, truncation=True, max_length=512)
    return tokenized_batch

dataset_enc = dataset.map(tokenize, batched=True, remove_columns=cols_to_remove, num_proc=4)

# Set dataset format for PyTorch
dataset_enc.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Check the output
print(dataset_enc["train"].column_names)
print(dataset_enc["val"].column_names)


Map (num_proc=4):   0%|          | 0/11984 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2996 [00:00<?, ? examples/s]

['labels', 'input_ids', 'attention_mask']
['labels', 'input_ids', 'attention_mask']


In [ ]:
# Instantiate a data collator with dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create data loaders for to reshape data for PyTorch model
train_dataloader = DataLoader(
    dataset_enc["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    dataset_enc["val"], batch_size=BATCH_SIZE, collate_fn=data_collator
)

In [ ]:
test_sample_ids =[i for i in test_df.PetID]

In [ ]:
# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")

# Load model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",
                                                           num_labels=num_labels) #19:11

Number of labels: 5


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Model parameters ---- codigo sin optuna
########OJO SIN OPTUNA



# learning_rate = 5e-5   #19.17
# num_epochs = 15


# # Create the optimizer
# optimizer = AdamW(model.parameters(), lr=learning_rate)    #aca hace el backpropágation 19.15
# #ver explicacion!!!!! para reducir el error-- el optimizador decide si hace descenso de gradiente o si tiene una inercia, etc..
# #no es lo mismo que optuna(ver otro cuaderno)

# # Further define learning rate scheduler
# num_training_batches = len(train_dataloader)
# num_training_steps = num_epochs * num_training_batches
# lr_scheduler = get_scheduler(
#     "linear",                   # linear decay
#     optimizer=optimizer,
#     num_warmup_steps=0,
#     num_training_steps=num_training_steps,
# )



**Miramos el Modelo**

In [ ]:

# Set the device automatically (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Move model to device
model.to(device)


 #768 features--- 19.13
#score que le da a todas la velocidad

# 19.14 (ventana de contexto) tamaño 512--- transoremen da tamaño maximo de texto que l epuede poner--- !!!
#se tokeniza la variable descripcion, no el dataset


cuda


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
MODEL_VERSION = '2.0'

In [ ]:
def train_val(model, dataloaders, datasets, device, num_epochs=4, lr=0.001, trial=None):

    since = time.time()

    # Create the optimizer
    optimizer = AdamW(model.parameters(), lr=lr)

    # Further define learning rate scheduler
    num_training_batches = len(train_dataloader)
    num_training_steps = num_epochs * num_training_batches
    lr_scheduler = get_scheduler(
        "linear",                   # linear decay
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )


    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_kappa =  -999

    train_losses = []
    val_losses = []

    try:
        previous_best = study.best_value
    except:
        previous_best = -999


    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        kappa_labels_true = []
        kappa_labels_predicted = []
        output_scores = []

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for batch in tqdm(dataloaders[phase]):
                batch = batch.to(device)
                #inputs = inputs.to(device)
                labels = batch.labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                # Track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(**batch)
                    loss = outputs.loss

                    preds = torch.nn.functional.softmax(outputs.logits, dim=-1)
                    preds_labels = torch.argmax(preds, dim=-1)


                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                    elif phase == 'val':
                        kappa_labels_true.extend(labels.cpu().numpy().tolist())
                        kappa_labels_predicted.extend(preds_labels.cpu().numpy().tolist())
                        outputs_np = preds.cpu().numpy()
                        output_scores.extend([outputs_np[i,:] for i in range(outputs_np.shape[0])])

                # Statistics
                running_loss += loss.item() * labels.size(0)
                running_corrects += torch.sum(preds_labels == labels.data)

                #END OF BATCH

            epoch_loss = running_loss / len(datasets[phase])
            epoch_acc = running_corrects.double() / len(datasets[phase])

            if phase == 'train':
                train_losses.append(epoch_loss)
                kappa_score = np.nan
            else:
                val_losses.append(epoch_loss)
                kappa_score = cohen_kappa_score(kappa_labels_true,
                                  kappa_labels_predicted,
                                  weights = 'quadratic')



            print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}% Kappa: {kappa_score:.3f}')

            # If this is the best Epoch so far -> Deep copy the model
            if phase == 'val' and kappa_score > best_kappa:
                best_acc = epoch_acc
                best_kappa = kappa_score
                best_model_wts = copy.deepcopy(model.state_dict())


                #Best Epoch within a trial and better than previous trials
                if trial is not None and best_kappa > previous_best:

                    #Save test dataset with predictions
                    predicted_filename = os.path.join(PATH_TO_TEMP_FILES,f'test_{trial.study.study_name}_{trial.number}.joblib')
                    predicted_df = pd.DataFrame({'PetID':test_sample_ids,
                                'pred':output_scores}).merge(test_df, on='PetID')
                    dump(predicted_df, predicted_filename)

                    #Generate and save CM
                    #cm_filename = os.path.join(PATH_TO_TEMP_FILES,f'cm_{trial.study.study_name}_{trial.number}.jpg')
                    #plot_confusion_matrix(kappa_labels_true,kappa_labels_predicted).write_image(cm_filename)

            #END OF PHASE

        #END OF EPOCH

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:.2f}%'.format(best_acc * 100))

    # Load best model weights
    model.load_state_dict(best_model_wts)

    # Save in optuna trial the best test dataset, cm and model weights
    if trial is not None and best_kappa > previous_best:
        upload_artifact(trial, predicted_filename, artifact_store)

        #upload_artifact(trial, cm_filename, artifact_store)

        file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{trial.number}.pth'
        model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
        torch.save(model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
        upload_artifact(trial, model_path, artifact_store)

    return model,best_kappa



In [ ]:

# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")


Number of labels: 5


In [ ]:
best_model,_ = train_val(model,
                       dataloaders={'train': train_dataloader,
                                    'val': eval_dataloader},
                       datasets=dataset_enc,
                       device=device,
                       lr = 5e-5,
                       num_epochs=15)

Epoch 0/14
----------


  0%|          | 0/188 [00:00<?, ?it/s]

Train Loss: 1.4454 Acc: 32.02% Kappa: nan


  0%|          | 0/47 [00:00<?, ?it/s]

Val Loss: 1.4186 Acc: 34.55% Kappa: 0.148
Epoch 1/14
----------


...

Val Loss: 3.2175 Acc: 36.85% Kappa: 0.238
Epoch 14/14
----------


  0%|          | 0/188 [00:00<?, ?it/s]

Train Loss: 0.1128 Acc: 95.57% Kappa: nan


  0%|          | 0/47 [00:00<?, ?it/s]

Val Loss: 3.4243 Acc: 37.35% Kappa: 0.240
Training complete in 157m 19s
Best val Acc: 38.52%


In [ ]:
# Guardo el modelo
run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{run_id}.pth'
model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
torch.save(best_model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
print(f'Modelo guardado en {model_path}')

Modelo guardado en /content/work/optuna_temp_artifacts/06 Bert_1.0_20260427_005440.pth


In [ ]:
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)

def optuna_train(trial):

    # Increased the range for epochs to allow for more training iterations
    epochs = trial.suggest_int('epochs', 5, 15)

    # Expanded the learning rate search range
    lr = trial.suggest_float('lr', 1e-5, 5e-4, log=True)

    _,best_score = train_val(model,
                       dataloaders={'train': train_dataloader,
                                    'val': eval_dataloader},
                       datasets=dataset_enc,
                       device=device,
                       num_epochs=epochs,
                       lr=lr,
                       trial=trial)


    return(best_score)

In [ ]:
import os

db_path = os.path.join(os.path.dirname(PATH_TO_OPTUNA_ARTIFACTS), 'db.sqlite3')
study = optuna.create_study(direction='maximize',
                            storage=f"sqlite:///{db_path}",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_1.0', #uso el original que dio mejor kappa
                            load_if_exists = True)
# Increased the number of trials to 10 for a more extensive search
study.optimize(optuna_train, n_trials=10)

[I 2026-04-30 01:22:49,511] Using an existing study with name '06 Bert_1.0' instead of creating a new one.


Epoch 0/11
----------


  0%|          | 0/188 [00:00<?, ?it/s]

Train Loss: 0.9187 Acc: 63.83% Kappa: nan


  0%|          | 0/47 [00:00<?, ?it/s]

...

  0%|          | 0/188 [00:00<?, ?it/s]

Train Loss: 0.3602 Acc: 87.01% Kappa: nan


  0%|          | 0/47 [00:00<?, ?it/s]

Val Loss: 2.5349 Acc: 35.65% Kappa: 0.231
Epoch 6/11
----------


  0%|          | 0/188 [00:00<?, ?it/s]

In [ ]:
print(f"Best value: {study.best_value}")
print(f"Best params: {study.best_params}")

Best value: 0.2413909520594194
Best params: {'epochs': 2, 'lr': 2.111300120524732e-05}


In [ ]:
best_trial = study.best_trial

if best_trial:
    best_model_filename = f'{MODEL_NAME}_{MODEL_VERSION}_2904_{best_trial.number}.pth'
    best_model_path = os.path.join(PATH_TO_TEMP_FILES, best_model_filename)
    torch.save(best_model_filename, best_model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
    print(f'Modelo guardado en {best_model_path}')


Modelo guardado en /content/work/optuna_temp_artifacts/06 Bert_2904_4.pth


**Entrenamos**

In [ ]:
###### apartir de aca es el codigo sin optuna. Reutilizar kappa

In [ ]:
progress_bar = tqdm(range(num_training_steps))

# Train the model with PyTorch training loop
model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)    #en maquina 20 min 19.18

  0%|          | 0/2820 [00:00<?, ?it/s]

**Obtenemos la kappa base**

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Inicializa listas para almacenar todas las predicciones y etiquetas
all_predictions = []
all_labels = []

# Iteratively evaluate the model and collect predictions and labels
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    # Mover predicciones y etiquetas a CPU y convertir a numpy
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

# Convertir listas a arrays de numpy
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Calcular Quadratic Weighted Kappa
qwk = cohen_kappa_score(all_labels, all_predictions, weights='quadratic')

print(f"Quadratic Weighted Kappa: {qwk}")


Quadratic Weighted Kappa: 0.23084809428919872


**Predecimos un ejemplo de descripción**

In [ ]:
# Un ejemplo
desc = test_df.iloc[4]['Description']
print(desc)

# Tokenize inputs
inputs = tokenizer(desc, padding=True, truncation=True, return_tensors="pt").to(device) # Move the tensor to the GPU

# Inference model and get logits
outputs = model(**inputs)

# Convert logits to class probabilities
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
probabilities = predictions.detach().cpu().numpy()
predicted_class = np.argmax(probabilities, axis=1)

# Establecer opciones de impresión para evitar la notación científica
np.set_printoptions(suppress=True, formatter={'float_kind': '{:.8f}'.format})
print(probabilities[0])
print(predicted_class)

#aca da ulas probalidadde de una descripvion

Cute active little puppy found abondoned and looking for a loving home to live. Please adopt me.....
[0.00005997 0.99879503 0.00101228 0.00009858 0.00003415]
[1]


In [ ]:
# Define el directorio donde quieres guardar el modelo
output_dir = './my_distilbert_model'

# Guarda el modelo y el tokenizer en el directorio especificado
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Modelo y tokenizer guardados en: {output_dir}")

# Para descargar a tu computadora local, busca la carpeta 'my_distilbert_model'
# en el explorador de archivos de Colab (panel izquierdo) y descárgala.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo y tokenizer guardados en: ./my_distilbert_model


PARA CARGAR MODELO ENTRENADO

In [ ]:
#para cargar el modelo entrenado!
from transformers import AutoModelForSequenceClassification, DistilBertTokenizerFast

# Define el directorio donde guardaste el modelo
output_dir = './my_distilbert_model' # O la ruta completa a la carpeta

# Carga el tokenizer
loaded_tokenizer = DistilBertTokenizerFast.from_pretrained(output_dir)

# Carga el modelo
loaded_model = AutoModelForSequenceClassification.from_pretrained(output_dir)

print("Modelo y tokenizer cargados exitosamente!")

# Ahora puedes usar loaded_tokenizer y loaded_model para hacer inferencias.